**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Numerical Linear Algebra

The most practically valuable math course in the curriculum: why solves explode, which factorization to use when, how Krylov methods solve systems too big to factor, and the randomized SVD that makes modern data sizes tractable — every method checked against LAPACK ground truth.

## 1. Pre-requisites

[Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) — this is its 'in floating point, at scale' sequel.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Conditioning & Stability* (~40 min)
**Goal:** separate the problem's sensitivity from the algorithm's sins; know when to distrust a solve.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (QR & least squares).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Conditioning & Stability</b></summary>

**Timing (~40 min).** 12 min separating the two failure modes · 10 min the digit-loss rule · 12 min the demo · 6 min the normal-equations preview.

**Board first — the distinction that organises the whole workshop.** **Conditioning is the problem's fault**; **stability is the algorithm's fault**. Write both definitions and insist on the difference: $\kappa(A) = \sigma_{\max}/\sigma_{\min}$ measures how much the *answer* moves when the *data* wiggles, and no algorithm however clever can beat it. A stable algorithm, separately, returns the exact answer to a *nearby* problem. Students who conflate these spend their careers blaming solvers for ill-posed questions.

**Make the digit-loss rule concrete.** You lose roughly $\log_{10}\kappa$ digits. Float64 gives about 16, so $\kappa = 10^8$ leaves 8 and $\kappa = 10^{16}$ leaves none. Have the room compute the surviving digits for a few $\kappa$ before running the cell — it converts an abstract number into a budget.

**Then be precise that $\kappa\varepsilon$ is a *bound*, not a prediction.** The measured errors come in 5–50× **below** the printed prediction at every $\kappa$. That is not the theory failing; worst-case analysis assumes the error aligns with the most sensitive singular direction, and a random right-hand side does not. Ask the room whether they would rather have a bound that is usually pessimistic or a typical-case estimate that occasionally lies — for numerical software the answer is the bound, because you need a guarantee. This is a good moment to teach reading an inequality as an inequality.

**Point at the $\kappa$ pun.** The same $\kappa$ that prices digits here sets gradient descent's convergence rate in [Optimization](../Optimization/Optimization.ipynb), and reappears in Session 3 as CG's $\sqrt\kappa$. One number, three roles — worth flagging so students recognise it later rather than meeting it three times as a stranger.

**Set up Session 2 with the teaser.** Solving least squares via the normal equations *squares* the condition number, since $\kappa(A^\top A) = \kappa(A)^2$. Ask what that does to a comfortable $\kappa = 10^8$: it becomes $10^{16}$, and every digit is gone. Do not resolve it — Session 2's Läuchli demo is much more satisfying if the room has been sitting with the problem.

**If you want a live demonstration**, raise $\kappa$ to $10^{16}$ in this cell and watch the relative error reach order 1: the computed answer becomes pure noise while `np.linalg.solve` reports no error at all. Silent catastrophic failure is the thing to fear, and seeing it once is worth more than the rule of thumb.
</details>

## 2. Two Different Ways to Be Wrong

💡 **Intuition.** **Conditioning** is the *problem's* fault: κ(A) = σ_max/σ_min measures how much the answer moves when the data wiggles — no algorithm can beat it. **Stability** is the *algorithm's* fault: a stable algorithm returns the exact answer to a nearby problem. The rule of thumb with teeth: you lose about $\log_{10}\kappa$ digits — solve with κ=10⁸ in float64 and only ~8 of your 16 digits survive. The [GD convergence κ](../Optimization/Optimization.ipynb) and this κ are the same number wearing two hats.

In [2]:
# watch digits die exactly on schedule
for kappa in [1e2, 1e6, 1e10, 1e14]:
    n_dim = 50
    U, _ = np.linalg.qr(rng.standard_normal((n_dim, n_dim)))
    V, _ = np.linalg.qr(rng.standard_normal((n_dim, n_dim)))
    s = np.logspace(0, -np.log10(kappa), n_dim)
    A = U @ np.diag(s) @ V.T
    x_true = rng.standard_normal(n_dim)
    x_hat = np.linalg.solve(A, A @ x_true)
    rel = np.linalg.norm(x_hat - x_true) / np.linalg.norm(x_true)
    print(f"κ = {kappa:.0e}: relative error {rel:.1e}   (predicted ~ κ·ε ≈ {kappa*2.2e-16:.1e})")

κ = 1e+02: relative error 4.3e-15   (predicted ~ κ·ε ≈ 2.2e-14)
κ = 1e+06: relative error 7.9e-12   (predicted ~ κ·ε ≈ 2.2e-10)
κ = 1e+10: relative error 4.4e-08   (predicted ~ κ·ε ≈ 2.2e-06)
κ = 1e+14: relative error 7.8e-04   (predicted ~ κ·ε ≈ 2.2e-02)


**What just happened.** Four solves, and the accuracy degrades on a schedule set entirely by $\kappa$:

| $\kappa$ | measured error | $\kappa\varepsilon$ bound | digits left |
|---|---|---|---|
| $10^{2}$ | 4.3e-15 | 2.2e-14 | ~14 |
| $10^{6}$ | 7.9e-12 | 2.2e-10 | ~11 |
| $10^{10}$ | 4.4e-08 | 2.2e-06 | ~7 |
| $10^{14}$ | 7.8e-04 | 2.2e-02 | ~3 |

Every $10^4$ increase in $\kappa$ costs about four digits. The rule "you lose $\log_{10}\kappa$ digits" is not a heuristic here — it is visibly the governing law, and it lets you predict your accuracy *before* solving anything.

**And nothing was wrong with the algorithm.** `np.linalg.solve` is backward stable, meaning it returns the exact answer to a nearby problem. The error is entirely the **problem's** sensitivity: at $\kappa = 10^{14}$, a perturbation the size of floating-point rounding moves the true answer by 1e-2, and no algorithm can do better. This is the distinction to keep: conditioning is the problem's fault, stability is the algorithm's fault, and confusing them leads people to blame solvers for ill-posed questions.

**Note that the measurements come in 5–50× below the printed bound**, consistently. That is not the theory being wrong — $\kappa\varepsilon$ is a **worst case**, attained when the perturbation aligns with the most sensitive singular direction. A random right-hand side does not align that way, so typical errors sit an order or two under the ceiling. Read the inequality as an inequality: the bound guarantees you will not do *worse*, and numerical software needs exactly that kind of guarantee even when it is usually pessimistic.

**The most dangerous feature of this table is what it does not contain: any warning.** Every one of those solves returned normally. No exception, no flag, no hint that the $\kappa = 10^{14}$ answer carries only three good digits. Silent loss of accuracy is the characteristic failure of numerical linear algebra, which is why computing $\kappa$ — or at least knowing roughly what it is — belongs in any pipeline that solves systems.

**A forward reference worth planting now.** This $\kappa$ is the same number that sets gradient descent's convergence rate in [Optimization](../Optimization/Optimization.ipynb), and it returns in Session 3 as CG's $\sqrt\kappa$. And the immediate sequel: solving least squares through the normal equations forms $A^\top A$, whose condition number is $\kappa^2$. A comfortable $\kappa = 10^8$ becomes $10^{16}$, and this table says every digit is gone. Session 2 shows that happening.

**The classic self-inflicted wound:** solving least squares via the normal equations *squares* the condition number ($\kappa(A^TA) = \kappa(A)^2$). The fix is Session 2.

---
### 🕐 Session 2 of 4 — *QR & Least Squares Done Right* (~35 min)
**Goal:** orthogonalization as the workhorse; Householder vs Gram-Schmidt; the κ² trap escaped.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Krylov).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: QR & Least Squares Done Right</b></summary>

**Timing (~35 min).** 8 min QR and why orthogonality helps · 10 min Gram–Schmidt vs Householder · 12 min the Läuchli kill shot · 5 min the rule.

**Board first — why orthogonal matrices are the safe currency.** An orthogonal $Q$ has $\kappa(Q) = 1$ exactly: it preserves lengths and angles, so multiplying by it neither amplifies errors nor loses information. That single property is why every stable factorisation in numerical linear algebra is built from orthogonal pieces. Then $A = QR$ turns least squares into $Rx = Q^\top b$ — a triangular solve at the **original** $\kappa$, not $\kappa^2$.

**Make the Gram–Schmidt comparison feel unfair, then explain why it isn't.** Both algorithms compute the same $Q$ in exact arithmetic. In floating point, classical Gram–Schmidt gives $\|Q^\top Q - I\| = 5.9\times10^{-8}$ while Householder gives $6.7\times10^{-16}$ — **eight orders of magnitude** apart on identical input. Ask what differs. Gram–Schmidt *subtracts* projections, and subtraction of nearly-equal quantities is where catastrophic cancellation lives; Householder *reflects*, and reflections are orthogonal transformations that cannot amplify error. Same mathematics, different arithmetic — and that gap is the soul of this course.

**Note the input is deliberately treacherous.** `A = u[:, None] + 1e-7 * randn(...)` builds 40 columns that are nearly identical. Say so: this is a stress test, not typical data, and on well-conditioned input Gram–Schmidt is fine. The lesson is that *you often cannot tell in advance*, which is why LAPACK simply always uses Householder.

**The Läuchli matrix is the session's best moment — set it up as a bet.** $\kappa(A) = 1.4\times10^8$, comfortably solvable in float64 with about 8 digits to spare. Ask the room to predict what the normal equations will do. Then reveal: $\kappa(A^\top A) = \infty$, `LinAlgError: Singular matrix`, **total failure**, while QR returns the exact answer to 2.2e-16.

**Explain precisely why $A^\top A$ becomes singular**, because "it squares $\kappa$" alone is not enough. With $\varepsilon = 10^{-8}$, forming $A^\top A$ requires computing $1 + \varepsilon^2 = 1 + 10^{-16}$ — and since machine epsilon is $2.2\times10^{-16}$, that sum **rounds to exactly 1**. The information distinguishing the columns is destroyed by a single addition, before any solving begins. The trap springs precisely when $\varepsilon < \sqrt{\varepsilon_{\text{machine}}}$, which is a checkable condition rather than folklore.

**Ask the room.** "The normal equations are in every statistics textbook. Are they wrong?" Not wrong — they are the correct *mathematics* and a poor *algorithm*. Mathematically $x = (A^\top A)^{-1}A^\top b$ is exactly the least-squares solution. Numerically it throws away half your digits before starting. Separating "what the answer is" from "how to compute it" is the distinction this whole workshop teaches.

**The rule to send them away with: never form $A^\top A$, never invert a matrix, always factor.** It is the same "factor, don't expand" principle as second-order sections in [Filter Design](../../Intro_DSP/Filter_Design.ipynb) and square-root [RLS](../../Intro_Time_Series/Intro_RLS.ipynb) updates.
</details>

## 3. Factor, Don't Invert

💡 **Intuition.** QR rewrites $A = QR$ (orthonormal × triangular): least squares becomes $R x = Q^T b$ — a stable triangular solve at the *original* κ, not κ². And not all orthogonalizations are equal: classical Gram–Schmidt loses orthogonality catastrophically in ill-conditioned bases; **Householder reflections** (LAPACK's choice) keep $Q^TQ = I$ to machine precision. Same math, different arithmetic — the whole soul of this course.

In [3]:
def gram_schmidt(A):
    Q = A.astype(float).copy()
    for j in range(A.shape[1]):
        for i in range(j):
            Q[:, j] -= (Q[:, i] @ Q[:, j]) * Q[:, i]
        Q[:, j] /= np.linalg.norm(Q[:, j])
    return Q

# a genuinely treacherous basis: 40 nearly-IDENTICAL columns
n_dim = 40
u = rng.standard_normal(n_dim)
A = u[:, None] + 1e-7 * rng.standard_normal((n_dim, n_dim))
Q_gs = gram_schmidt(A)
Q_hh, _ = np.linalg.qr(A)                            # Householder under the hood
print(f"‖QᵀQ − I‖  Gram-Schmidt: {np.abs(Q_gs.T @ Q_gs - np.eye(n_dim)).max():.1e}")
print(f"‖QᵀQ − I‖  Householder:  {np.abs(Q_hh.T @ Q_hh - np.eye(n_dim)).max():.1e}")

‖QᵀQ − I‖  Gram-Schmidt: 5.9e-08
‖QᵀQ − I‖  Householder:  6.7e-16


**What just happened.** Identical input, identical mathematics, and **eight orders of magnitude** difference in the result: Gram–Schmidt loses orthogonality at $5.9\times10^{-8}$ while Householder holds it at $6.7\times10^{-16}$ — machine precision.

In exact arithmetic these two algorithms compute the *same* $Q$. The entire gap is floating point, and the mechanism is worth naming precisely. Classical Gram–Schmidt **subtracts** projections: `Q[:, j] -= (Q[:, i] @ Q[:, j]) * Q[:, i]`. When the columns are nearly parallel, that subtracts two nearly-equal quantities, and the leading digits cancel — leaving a result dominated by rounding error in the digits that survive. That is **catastrophic cancellation**, and each subsequent column inherits and compounds the damage.

Householder never subtracts nearly-equal things. It builds $Q$ from **reflections**, which are themselves orthogonal transformations, and an orthogonal transformation cannot amplify error — its condition number is exactly 1. So error stays where it started instead of growing. Same mathematics, different arithmetic, and that distinction is the soul of numerical linear algebra.

**A caveat on the input, so nobody over-generalises.** `A = u[:, None] + 1e-7 * randn(...)` constructs 40 columns that are nearly identical — a deliberately treacherous basis. On well-conditioned input Gram–Schmidt behaves acceptably, and the 5.9e-8 here is a stress-test result, not typical performance.

But the practical conclusion survives that caveat, because **you usually cannot tell in advance** whether your basis is treacherous. That is exactly why LAPACK — and therefore `np.linalg.qr` — always uses Householder, and why "just implement Gram–Schmidt, it is three lines" is one of the more expensive shortcuts available in scientific computing. (Modified Gram–Schmidt is meaningfully better than the classical version shown here, and still not as good as Householder.)

**Why orthogonality mattering so much.** An orthogonal matrix has $\kappa = 1$ exactly: it preserves lengths and angles, amplifying nothing. That is what makes $A = QR$ safe — least squares becomes the triangular solve $Rx = Q^\top b$ at the *original* condition number rather than the squared one. Losing orthogonality in $Q$ means losing precisely the property the factorisation was chosen for. The next cell shows what happens when you give it up entirely.

In [4]:
# The κ² kill shot: Läuchli's matrix (κ ≈ 1.4e8, comfortably solvable — at κ, not κ²)
eps = 1e-8                                            # < √(machine epsilon): the trap springs
A = np.array([[1.0, 1.0], [eps, 0.0], [0.0, eps]])
x_ref = np.array([1.0, 1.0])                          # KNOWN solution (the oracle)
b = A @ x_ref

print(f"cond(A)    = {np.linalg.cond(A):.1e}   ← fine for float64")
print(f"cond(AᵀA)  = {np.linalg.cond(A.T @ A):.1e}   ← κ² rounded AᵀA to exactly singular!")
try:
    x_normal = np.linalg.solve(A.T @ A, A.T @ b)
    print("normal equations:", x_normal)
except np.linalg.LinAlgError as e:
    print(f"normal equations: LinAlgError ({e}) — total failure")
Q, R = np.linalg.qr(A)
x_qr = np.linalg.solve(R, Q.T @ b)
print(f"QR:               {x_qr}   (exact — error {np.abs(x_qr - x_ref).max():.1e})")

cond(A)    = 1.4e+08   ← fine for float64
cond(AᵀA)  = inf   ← κ² rounded AᵀA to exactly singular!
normal equations: LinAlgError (Singular matrix) — total failure
QR:               [1. 1.]   (exact — error 2.2e-16)


**What just happened.** The same least-squares problem, two methods, and one of them **failed completely**:

- $\kappa(A) = 1.4\times10^8$ — comfortably solvable in float64, with about 8 digits to spare.
- $\kappa(A^\top A) = \infty$ — the normal equations matrix came out **exactly singular**.
- Normal equations: `LinAlgError: Singular matrix`. No answer at all.
- QR: `[1. 1.]`, the exact known solution, error 2.2e-16.

**Work out precisely how $A^\top A$ became singular**, because "it squares the condition number" understates what happened. With $\varepsilon = 10^{-8}$, forming $A^\top A$ requires computing $1 + \varepsilon^2 = 1 + 10^{-16}$. Machine epsilon is $2.2\times10^{-16}$, so **that sum rounds to exactly 1**. The off-diagonal entry is also exactly 1, giving a matrix of all ones — singular, rank 1.

The information distinguishing the two columns was destroyed by a *single addition*, before any solving began. Not degraded, not ill-conditioned: erased. And the trap has a checkable trigger — it springs when $\varepsilon < \sqrt{\varepsilon_{\text{machine}}} \approx 1.5\times10^{-8}$, which is exactly where this example was placed.

**QR never forms that product.** It factors $A = QR$ directly and solves $Rx = Q^\top b$, working at $\kappa = 1.4\times10^8$ rather than $\kappa^2$. Same problem, same precision, exact answer. The difference is entirely in the route taken to it.

**So: are the normal equations wrong?** No — and this distinction is the point of the whole workshop. $x = (A^\top A)^{-1}A^\top b$ is the *correct mathematics*; it is exactly the least-squares solution, and every statistics textbook is right to derive it. It is a **poor algorithm**, because it discards half your available precision before the solve starts. Knowing what the answer *is* and knowing how to *compute* it are different skills, and numerical analysis is the second one.

**The rule to take away: never form $A^\top A$, never invert a matrix, always factor.** `np.linalg.lstsq` uses SVD, `scipy.linalg.lstsq` offers QR — both avoid the trap by default, and the only way to fall into it is to write the textbook formula literally.

This is the same "factor, don't expand" principle that appears as second-order sections in [Filter Design](../../Intro_DSP/Filter_Design.ipynb) and square-root forms in [RLS](../../Intro_Time_Series/Intro_RLS.ipynb). Whenever a quantity must retain a structural property — orthogonality, positive-definiteness, a pole inside the unit circle — keep it in factored form, because the factors cannot silently violate what the expanded object can.

One thing to appreciate about this demo's design: `x_ref = [1, 1]` is chosen *first* and `b = A @ x_ref` constructed from it. The answer is known exactly, so "QR error 2.2e-16" is a genuine oracle check rather than a comparison against another possibly-wrong computation.

---
### 🕐 Session 3 of 4 — *Krylov Methods: Conjugate Gradients* (~40 min)
**Goal:** solve systems you can only MULTIPLY by; watch κ set the convergence rate.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (randomized SVD).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Krylov Methods — Conjugate Gradients</b></summary>

**Timing (~40 min).** 8 min when factorisation is impossible · 12 min the Krylov subspace idea · 12 min the demo and its two very different outcomes · 8 min preconditioning.

**Board first — establish that factorisation sometimes is not an option.** A graph Laplacian for a social network, or a discretised PDE on a fine grid, can be $10^6 \times 10^6$. Storing a dense factor is $10^{12}$ numbers; it does not fit and never will. But you *can* compute $Av$ cheaply, because $A$ is sparse. Ask what can be built from nothing but matrix–vector products — and the answer is the Krylov subspace $\mathrm{span}\{b, Ab, A^2b, \dots\}$, which is exactly what repeated multiplication hands you for free.

**Then the reframe.** Krylov methods do not solve the system; they find the **best answer available inside that subspace**, and grow the subspace one product at a time. CG is the version for symmetric positive-definite $A$, and it is optimal in the $A$-norm over that subspace at every step — not a heuristic.

**The $\sqrt\kappa$ is the headline result — make sure it lands.** Error contracts like $\left(\frac{\sqrt\kappa-1}{\sqrt\kappa+1}\right)^k$. Compare with plain gradient descent's $\frac{\kappa-1}{\kappa+1}$: at $\kappa = 10^4$, gradient descent's rate is 0.9998 while CG's is 0.9802 — CG needs roughly **100× fewer iterations**. Ask where else a $\sqrt\kappa$ has appeared; it is the same acceleration momentum provides in [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb), and the connection is not a coincidence.

**Use the demo's two outcomes deliberately — they look inconsistent and are not.** At $\kappa = 10^2$ CG matches the direct solve to 2.7e-15; at $\kappa = 10^4$ it is only 2.9e-3 after the same 250 iterations. That is not a failure. The predicted contraction at $\kappa = 10^4$ is $0.9802^{250} = 6.7\times10^{-3}$, and the measured 2.9e-3 sits right on it. **CG is performing exactly to specification in both cases** — the specification is just much weaker at higher $\kappa$. Have the room compute $\text{rate}^{250}$ before revealing the second number; predicting a non-converged result from theory is more satisfying than seeing a clean one.

**Which motivates preconditioning properly.** If the rate is set by $\kappa$, the way to converge faster is not a better algorithm but a *better-conditioned problem*: solve $M^{-1}Ax = M^{-1}b$ with $M \approx A$ but cheap to invert. Preconditioning is where the real engineering in large-scale linear algebra lives, and it is worth saying that choosing $M$ is problem-specific craft rather than a library call.

**Contrast the two paradigms explicitly at the close.** Direct methods (Sessions 1–2) give the answer in a fixed number of operations and need the matrix. Iterative methods give a *sequence of improving answers* and need only products — so you stop when accuracy suffices, and $\kappa$ decides whether that is 10 iterations or 10,000. Knowing which regime you are in is the practical skill.
</details>

## 4. Solving Without Factoring

💡 **Intuition.** When $A$ is huge and sparse (a [graph Laplacian](../../Intro_DSP/Graph_Signal_Processing.ipynb)!, a discretized PDE), you can afford $Av$ products but never a factorization. **Krylov methods** build the best answer inside $\mathrm{span}\{b, Ab, A^2b, \dots\}$ — the subspace that matrix-multiplies give you for free. **CG** (for SPD $A$) is its masterpiece: each iteration one product, error contracting like $(\frac{\sqrt\kappa - 1}{\sqrt\kappa + 1})^k$ — *√κ, not κ* ([momentum's](../../Intro_Mach_Learn/Training_Dynamics.ipynb) secret sibling). Preconditioning = warping the problem to shrink κ before you start.

In [5]:
def cg(Amul, b, iters):
    x = np.zeros_like(b); r = b.copy(); p = r.copy()
    errs = [np.linalg.norm(r)]
    for _ in range(iters):
        Ap = Amul(p)
        alpha = (r @ r) / (p @ Ap)
        x += alpha * p
        r_new = r - alpha * Ap
        beta = (r_new @ r_new) / (r @ r)
        p = r_new + beta * p; r = r_new
        errs.append(np.linalg.norm(r))
    return x, np.array(errs)

n_dim = 400
plt.figure(figsize=(8, 3))
for kappa in [1e2, 1e4]:
    Qm, _ = np.linalg.qr(rng.standard_normal((n_dim, n_dim)))
    s = np.logspace(0, -np.log10(kappa), n_dim)
    A = Qm @ np.diag(s) @ Qm.T                      # SPD with known κ
    b = rng.standard_normal(n_dim)
    x_cg, errs = cg(lambda v: A @ v, b, 250)
    rate = (np.sqrt(kappa)-1)/(np.sqrt(kappa)+1)
    plt.semilogy(errs/errs[0], label=f"κ={kappa:.0e} (theory rate {rate:.3f}/iter)")
    # ORACLE: CG's answer vs direct solve
    print(f"κ={kappa:.0e}: ‖x_CG − x_direct‖/‖x‖ = "
          f"{np.linalg.norm(x_cg - np.linalg.solve(A, b))/np.linalg.norm(x_cg):.1e}")
plt.legend(); plt.xlabel("iteration"); plt.ylabel("relative residual")
plt.title("CG: √κ convergence — and never a factorization in sight")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

κ=1e+02: ‖x_CG − x_direct‖/‖x‖ = 2.7e-15
κ=1e+04: ‖x_CG − x_direct‖/‖x‖ = 2.9e-03


/tmp/ipykernel_2977446/3278043996.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


**What just happened.** Two runs of the same solver, and the results look inconsistent until you check them against theory:

- $\kappa = 10^2$: CG matches the direct solve to **2.7e-15** — machine precision.
- $\kappa = 10^4$: CG is only within **2.9e-3** after the same 250 iterations.

**Neither is a failure. Both are exactly on schedule.** CG's error contracts like $\left(\frac{\sqrt\kappa-1}{\sqrt\kappa+1}\right)^k$. At $\kappa = 10^4$ that rate is $99/101 = 0.9802$, so after 250 iterations the predicted residual is $0.9802^{250} = 6.7\times10^{-3}$ — and we measured 2.9e-3, sitting right on it. At $\kappa = 10^2$ the rate is $9/11 = 0.818$, and $0.818^{250}$ is far below machine precision, so convergence is complete long before iteration 250.

The lesson is that **$\kappa$ does not merely affect how fast CG converges — it decides whether 250 iterations is generous or hopeless.** Same algorithm, same budget, two orders of magnitude difference in $\kappa$, and one run finishes while the other is a third of the way there.

**Why $\sqrt\kappa$ is the headline.** Plain gradient descent contracts at $\frac{\kappa-1}{\kappa+1}$; CG at $\frac{\sqrt\kappa-1}{\sqrt\kappa+1}$. At $\kappa = 10^4$ that is 0.9998 against 0.9802 — CG needs roughly **100× fewer iterations** for the same accuracy. The square root is the entire value of the method, and it is the same acceleration momentum buys in [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb). Not an analogy: momentum and CG are closely related, and both extract a $\sqrt\kappa$ from the same structure.

**What CG never does is factor.** Look at the function signature — `cg(Amul, b, iters)` takes a *function* that multiplies by $A$, not $A$ itself. That is the whole point: for a $10^6\times10^6$ graph Laplacian or discretised PDE, a dense factor would be $10^{12}$ numbers and will never fit, but $Av$ is cheap because the matrix is sparse. Krylov methods build the best answer available in $\mathrm{span}\{b, Ab, A^2b, \dots\}$ — the subspace repeated multiplication hands you for free — and CG is optimal within it at every step, in the $A$-norm.

**Which is what makes preconditioning the real engineering.** If the rate is set by $\kappa$, you speed things up not with a better algorithm but with a better-conditioned *problem*: solve $M^{-1}Ax = M^{-1}b$ for some $M \approx A$ that is cheap to invert. Choosing $M$ is problem-specific craft — incomplete Cholesky, multigrid, domain-specific approximations — and it is where large-scale linear algebra actually lives.

**And the paradigm shift is worth naming.** Sessions 1–2 used *direct* methods: a fixed operation count, the matrix required, an exact answer at the end. This session is *iterative*: a sequence of improving answers, only products required, and you stop when accuracy suffices. The $\kappa = 10^4$ run above is not broken — it is a partial answer, which for many applications is entirely adequate and is available at a cost no direct method could match.

---
### 🕐 Session 4 of 4 — *Randomized SVD* (~35 min)
**Goal:** sketch first, factor later: near-best low-rank approximations at a fraction of the cost.
**Builds on:** Session 3; [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S4.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Randomized SVD</b></summary>

**Timing (~35 min).** 8 min the key insight · 10 min the algorithm · 10 min the demo · 7 min why randomness is safe.

**Board first — the insight, stated before any algorithm.** To find a rank-$k$ approximation you do not need all of $A$; you need its **range** — the $k$-dimensional subspace where its action lives. So the question becomes: how do you find a $k$-dimensional subspace cheaply? Answer: hit $A$ with $k+p$ random vectors and see where they land. A random vector has a component in every direction, so $A\Omega$ almost certainly spans nearly the top subspace.

**Address the discomfort directly, because students have it.** "Random" sounds like "unreliable." It is the opposite here: for the sketch to *fail*, all $k+p$ random vectors would have to be nearly orthogonal to the top subspace simultaneously — a probability that decays exponentially in the oversampling $p$. This is [concentration of measure](../Concentration/Concentration_Inequalities.ipynb) again, and it is also why [Random Matrix Theory](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb) sits next door in the curriculum. Randomised algorithms of this kind are more reliable than many deterministic heuristics, not less.

**Walk the four steps.** Sketch $Y = A\Omega$; optionally sharpen with a power iteration; orthonormalise with QR (Session 2's workhorse, doing the retraction job again); project to a small $(k+p)\times n$ core and SVD *that*. The cost is $O(mnk)$ against the full SVD's $O(mn\min(m,n))$, and the saving is exactly the ratio $k/\min(m,n)$.

**Explain the power iteration**, since it looks like a magic line. $A(A^\top Y)$ raises the singular values to the third power in the sketch, which exaggerates the gap between the top-$k$ and the tail, so the captured subspace is cleaner. It matters most when the spectrum decays slowly; with a sharp gap (as here) even `power=0` would do well.

**Frame the three printed numbers as different kinds of claim.** Singular values matching to 9.9e-15 is an *accuracy* claim. Reconstruction-error ratio 1.0000 against Eckart–Young optimal is an *optimality* claim — we are not merely close, we are indistinguishable from the best possible rank-20 approximation. And 55× is a *cost* claim. Students should notice that the first two make the third meaningful; a fast method that lost accuracy would be a different trade entirely.

**Be honest about what makes this instance easy.** The matrix is exactly rank-20 plus small noise, so the spectral gap is enormous and the sketch captures the range almost perfectly. On a matrix with slowly decaying singular values the accuracy would be visibly worse, more oversampling and more power iterations would be needed, and the ratio would exceed 1.0000. Say so — the method is genuinely excellent and this demo is its best case.

**Close on the wider point.** Randomised numerical linear algebra is one of the more consequential developments of the last two decades, and it underpins practical PCA on datasets that could not otherwise be touched. It is the same instinct as sketching in streaming algorithms: pay a vanishing probability of a bad answer in exchange for an order-of-magnitude cost reduction.
</details>

## 5. The Randomized Revolution

💡 **Intuition.** To find a rank-$k$ approximation you don't need all of $A$ — you need its **range**. Hit $A$ with $k{+}p$ random vectors: with overwhelming probability ([concentration](../Concentration/Concentration_Inequalities.ipynb)!) the sketch $Y = A\Omega$ spans nearly the same top subspace. Orthonormalize the sketch, project, SVD the small core: $O(mnk)$ instead of $O(mn\min(m,n))$, and accuracy within a hair of Eckart–Young optimal.

In [6]:
def rsvd(A, k, p=8, power=1):
    Omega = rng.standard_normal((A.shape[1], k + p))
    Y = A @ Omega                                     # capture the range from a sketch
    for _ in range(power):                            # one power iteration sharpens the subspace
        Y = A @ (A.T @ Y)
    Q, _ = np.linalg.qr(Y)
    B = Q.T @ A                                       # small (k+p) × n core
    Ub, s, Vt = np.linalg.svd(B, full_matrices=False)
    return Q @ Ub[:, :k], s[:k], Vt[:k]

# a big low-rank-plus-noise matrix
m, n_c, k = 1500, 1200, 20
A = rng.standard_normal((m, k)) @ rng.standard_normal((k, n_c)) + 0.05*rng.standard_normal((m, n_c))

tic = time.perf_counter(); U_f, s_f, Vt_f = np.linalg.svd(A, full_matrices=False); t_full = time.perf_counter()-tic
tic = time.perf_counter(); U_r, s_r, Vt_r = rsvd(A, k); t_rand = time.perf_counter()-tic

# ORACLE: singular values must match the full SVD
print("max relative error in top-20 singular values:", np.abs(s_r - s_f[:k]).max()/s_f[0])
err_opt  = np.linalg.norm(A - U_f[:, :k]*s_f[:k] @ Vt_f[:k])
err_rand = np.linalg.norm(A - U_r*s_r @ Vt_r)
print(f"‖A − Â_k‖: optimal {err_opt:.1f}   randomized {err_rand:.1f}   (ratio {err_rand/err_opt:.4f})")
print(f"time: full SVD {t_full*1e3:.0f} ms   randomized {t_rand*1e3:.0f} ms   ({t_full/t_rand:.0f}x)")

max relative error in top-20 singular values: 9.900614752923446e-15
‖A − Â_k‖: optimal 66.0   randomized 66.0   (ratio 1.0000)
time: full SVD 252 ms   randomized 5 ms   (55x)


**What just happened.** Three numbers, and they make three genuinely different claims:

- **Accuracy**: the top-20 singular values match the full SVD to **9.9e-15** — machine precision.
- **Optimality**: reconstruction error 66.0 against an Eckart–Young optimal 66.0, ratio **1.0000**. Not merely close to good; indistinguishable from the *best possible* rank-20 approximation.
- **Cost**: 5 ms against 252 ms — **55× faster**.

The ordering matters. A fast method that lost accuracy would be a different trade, and one worth arguing about. Here there is nothing to argue about: same answer, 55× less time.

**Why a random sketch captures the range.** To build a rank-$k$ approximation you do not need all of $A$ — you need the $k$-dimensional subspace where its action lives. Hit $A$ with $k+p$ random vectors and, because a random vector has a component along *every* direction, the images $A\Omega$ land almost entirely inside the top subspace. Orthonormalise, project, and SVD the small core.

**And "random" here means reliable, not risky.** For the sketch to fail, all 28 random vectors would have to be nearly orthogonal to the top-20 subspace *simultaneously*. That probability decays exponentially in the oversampling $p$, which is [concentration of measure](../Concentration/Concentration_Inequalities.ipynb) doing the work — the same machinery that made [Random Matrix Theory](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb)'s spectra deterministic. Randomised algorithms of this kind are frequently *more* dependable than deterministic heuristics, because their failure probability is quantifiable rather than merely unobserved.

Note the power iteration too: `Y = A @ (A.T @ Y)` effectively cubes the singular values in the sketch, exaggerating the gap between the top-$k$ and the tail so the captured subspace is cleaner. It matters most when the spectrum decays slowly.

**Now the honest caveat, because this is the method's best case.** `A` is *exactly* rank-20 plus small noise, so the spectral gap is enormous and the sketch captures the range almost perfectly — which is why the ratio came out at 1.0000 rather than 1.01 or 1.05. On a matrix with slowly decaying singular values, and no clean gap, you would need more oversampling and more power iterations, and the ratio would be visibly above 1. The method remains excellent there; it simply is not free.

**And note where the saving comes from.** The full SVD costs $O(mn\min(m,n))$; the randomised version costs $O(mnk)$. With $m = 1500$, $n = 1200$, $k = 20$, the predicted ratio is roughly $1200/20 = 60$ — against a measured 55×, which is close given Python overhead. **The speedup is not a trick, it is the ratio $k/\min(m,n)$**, so it grows as your matrix gets bigger relative to the rank you need. That is exactly why this technique made PCA practical on datasets that could not otherwise be touched.

## 6. Conclusion

κ prices every digit; QR dodges the κ² trap Gram–Schmidt falls into; CG solves at √κ per iteration with only matrix-multiplies; random sketches capture ranges with probability on your side. This is the difference between knowing linear algebra and *computing* it.

---
## Where next

- [Random Matrix Theory](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb) — why those random sketches work so well.
- [Graph Signal Processing](../../Intro_DSP/Graph_Signal_Processing.ipynb) — Krylov's natural habitat.
- [Scaling Neural Networks](../../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — the same flop-counting instincts, applied to training.